# Pulso Penonomé: qué pasó en Penonomé y qué dicen los datos

Análisis principal del proyecto del equipo RacoonCity. Sigue el flujo completo de un análisis de datos sobre el distrito de Penonomé (Coclé, Panamá) antes y después del terremoto del 9 de octubre de 2026, con lo aprendido en las semanas 1 a 6 del módulo de Python.

[Proyecto](README.md) | [Estudio completo](documento/estudio-penonome.md) | [Anexo: técnicas por semana](notebooks/anexo-semanas/s01-listas-diccionarios-conjuntos.ipynb)

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## 1. La pregunta

Penonomé es la capital de Coclé: 104,326 habitantes en 16 corregimientos (Censo 2023). El 9 de octubre de 2026 un sismo de magnitud 7.7 con epicentro en Azuero dañó su catedral, su mercado, su hospital y cientos de viviendas. La próxima encuesta de empleo con datos posteriores al sismo llega en 2027, así que hoy nadie puede medir el golpe directamente.

El análisis responde tres preguntas con datos oficiales verificados:

1. **¿Cómo estaba Penonomé antes del sismo?** Población, pobreza, carencias de las viviendas, economía y empleo.
2. **¿Qué le pasó?** Daños oficiales, réplicas y dónde se reportaron daños.
3. **¿Qué viene y dónde ayudar primero?** Qué enseña el sismo de Colombia de agosto de 2026, escenarios de desempleo y un índice de prioridad por corregimiento.

## 2. El flujo de análisis

```mermaid
flowchart LR
    A[Fuentes oficiales<br/>INEC, SINAPROC, ATP, DANE] --> B[Carga<br/>csv y pandas]
    B --> C[Limpieza y verificación<br/>sumas de control]
    C --> D[Análisis<br/>antes, sismo, después]
    D --> E[Visualización<br/>gráficas]
    E --> F[Decisión<br/>dónde ayudar primero]
```

| Paso | Qué se hace | Lo aprendido que se usa |
|---|---|---|
| Carga | Leer las tablas oficiales | Módulos y paquetes (semana 5), `pandas.read_csv` (semana 5) |
| Limpieza | Unir tablas, tipos, valores faltantes, controles | Diccionarios y conjuntos (semana 1), funciones (semana 2), `merge` y datos ordenados (semana 6) |
| Análisis | Tasas, cambios, comparaciones | Funciones y lambda (semana 2), `groupby` y series temporales (semana 6) |
| Prioridad | Ordenar corregimientos por necesidad | Ordenamiento por fusión (semana 4), búsqueda binaria (semana 3) |
| Visualización | Gráficas para el estudio y la presentación | matplotlib (semanas 5 y 6) |

Las técnicas se explican una por una en el anexo (`notebooks/anexo-semanas/`, de `s01` a `s06`); aquí se usan para contar la historia de Penonomé.

## 3. Carga

Todo el código reutilizable vive en el paquete `penonome` (`src/penonome/`). El módulo `pulso` carga las tablas del análisis desde `datos/`: los datos oficiales de Penonomé (`datos/penonome/`), la comparación con Colombia (`datos/comparacion-colombia/`) y las tablas propias del análisis (`datos/analisis/`).

**Lo aprendido:** módulos y paquetes ([unidad 34](../../01-python/semana-05/clase-2-modulos-y-pandas/u34-modulos-de-python.ipynb)) y DataFrames ([unidad 36](../../01-python/semana-05/clase-2-modulos-y-pandas/u36-dataframes-de-pandas.ipynb)).

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from penonome import pulso
from penonome.pulso import AZUL, NARANJA, AGUA, GRIS, TINTA2, estilo, figura, guardar
from penonome.ordenamiento import fusion
from penonome.busqueda import bin_search

t = pulso.cargar_todo()
for nombre, tabla in t.items():
    print(f"{nombre:18} {tabla.shape[0]:>4} filas x {tabla.shape[1]:>2} columnas")

catalogo             16 filas x  5 columnas
poblacion            16 filas x 10 columnas
viviendas            16 filas x 27 columnas
viviendas_totales     2 filas x 27 columnas
ipm                  16 filas x  2 columnas
pib                  20 filas x  8 columnas
desempleo             7 filas x  9 columnas
danos                 3 filas x  5 columnas
registros            16 filas x  6 columnas
replicas              7 filas x  3 columnas
eml_2026             29 filas x  8 columnas
poblacion_proy        6 filas x  6 columnas
situacion            17 filas x  9 columnas
geih                104 filas x 13 columnas
escenarios            5 filas x  4 columnas


## 4. Limpieza y verificación

Antes de analizar se comprueba que los datos cuadren. Si una suma no da, el error está en los datos y no en el análisis: así se encontró que la fila "Extraterritoriales" de Coclé traía un valor de Colón.

**Lo aprendido:** diccionarios y conjuntos ([unidades 14 a 16](../../01-python/semana-01/clase-2-diccionarios-y-conjuntos/u16-conjuntos.ipynb)), funciones ([unidad 17](../../01-python/semana-02/clase-1-funciones-recursividad-lambda/u17-funciones.ipynb)) y `merge` de pandas ([unidad 37](../../01-python/semana-06/clase-1-ordenacion-y-series-temporales/u37-ordenacion-de-datos.ipynb)).

In [3]:
corr = pulso.tabla_corregimientos(t)
distrito = t["viviendas_totales"].set_index("nivel").loc["distrito"]

# Conjuntos: ¿los 16 corregimientos aparecen en todas las tablas que se unieron?
nombres = {"catálogo": set(t["catalogo"]["corregimiento"]), "pobreza": set(t["ipm"]["corregimiento"]),
           "daños": set(t["registros"]["corregimiento"])}
faltan = {k: nombres["catálogo"] - v for k, v in nombres.items()}
print("Corregimientos sin pareja en alguna tabla:", {k: v for k, v in faltan.items() if v} or "ninguno")
print("Valores faltantes tras unir:", int(corr.isna().sum().sum()))

def tasa(parte, total):
    """Porcentaje con un decimal."""
    return round(parte / total * 100, 1)

verificar("los 16 corregimientos suman la población del distrito", int(corr["pob_2023"].sum()), 104326)
verificar("las viviendas de los corregimientos suman las del distrito", int(corr["viviendas"].sum()), int(distrito["viviendas"]))
verificar("la unión no pierde corregimientos", len(corr), 16)
d = t["desempleo"]
verificar("ocupados + desocupados = PEA en los 7 cortes de la EML",
          bool((d["ocupados_cocle"] + d["desocupados_cocle"] == d["pea_cocle"]).all()), True)
verificar("la tasa publicada coincide con desocupados / PEA",
          [tasa(a, b) for a, b in zip(d["desocupados_cocle"], d["pea_cocle"])], d["tasa_cocle"].tolist())

Corregimientos sin pareja en alguna tabla: ninguno
Valores faltantes tras unir: 0
Correcto: los 16 corregimientos suman la población del distrito
Correcto: las viviendas de los corregimientos suman las del distrito
Correcto: la unión no pierde corregimientos
Correcto: ocupados + desocupados = PEA en los 7 cortes de la EML
Correcto: la tasa publicada coincide con desocupados / PEA


## 5. ¿Cómo estaba Penonomé antes del sismo?

### 5.1 ¿Dónde vive la gente y dónde está la pobreza?

La población se concentra en el sur urbano (la cabecera, El Coco y Cañaveral), mientras la pobreza se concentra en el norte rural y montañoso. Uno de los 16 corregimientos se llama Coclé, como la provincia; en las gráficas aparece como Coclé (corregimiento). El gráfico ordena los corregimientos por población y los colorea por su pobreza multidimensional de 2023.

**Lo aprendido:** ordenamiento por fusión ([unidad 28](../../01-python/semana-04/clase-1-algoritmos-de-ordenamiento/u28-ordenamiento-por-fusion.ipynb)) para ordenar los corregimientos.

In [4]:
filas = corr.to_dict("records")
ordenados, comparaciones = fusion(filas, clave=lambda c: c["pob_2023"])
verificar("la fusión ordena igual que sorted()", [c["corregimiento"] for c in ordenados],
          [c["corregimiento"] for c in sorted(filas, key=lambda c: c["pob_2023"])])

fig, ax = figura(10, 5.8)
colores = [NARANJA if c["ipm"] >= 45 else AZUL for c in ordenados]
barras = ax.barh([c["corregimiento"] for c in ordenados], [c["pob_2023"] for c in ordenados], color=colores, height=0.7)
for b, c in zip(barras, ordenados):
    ax.text(b.get_width() + 250, b.get_y() + b.get_height() / 2, f"{c['pob_2023']:,}  (pobreza {c['ipm']:.0f} %)",
            va="center", fontsize=8.5, color=TINTA2)
ax.set_xlim(0, 33000)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=AZUL), plt.Rectangle((0, 0), 1, 1, color=NARANJA)],
          labels=["Pobreza multidimensional menor a 45 %", "45 % o más"], frameon=False, loc="lower right")
estilo(ax, "Penonomé: la gente vive en el sur y la pobreza está en el norte",
       "INEC, Censo 2023, Cuadro 4; PNUD/MEF, IPM por corregimiento 2023.", xlabel="Habitantes en 2023", rejilla="x")
guardar(fig, "pulso-01-poblacion-pobreza")
plt.show()

sur = corr[corr["ipm"] < 45]
print(f"Sur (pobreza < 45 %): {len(sur)} corregimientos, {sur['pob_2023'].sum():,} habitantes "
      f"({tasa(sur['pob_2023'].sum(), 104326)} % del distrito)")

Correcto: la fusión ordena igual que sorted()
Sur (pobreza < 45 %): 6 corregimientos, 64,036 habitantes (61.4 % del distrito)


/var/folders/xp/xx59bb5s1n31zc4kh2lnljyc0000gn/T/ipykernel_70040/725270340.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** seis corregimientos del sur reúnen al 61 % de la población con pobreza baja; los diez del norte tienen pobreza de 45 % a 86 %.

### 5.2 ¿Qué carencias tenían las viviendas?

Con el Censo 2023 se comparan las carencias de las viviendas del distrito, de Coclé y de la República.

**Lo aprendido:** funciones que reciben una tabla y devuelven porcentajes ([unidad 17](../../01-python/semana-02/clase-1-funciones-recursividad-lambda/u17-funciones.ipynb)) y `map` con `lambda` ([unidad 19](../../01-python/semana-02/clase-1-funciones-recursividad-lambda/u19-lambda-filter-map-reduce.ipynb)).

In [5]:
# Fila de la República en el Censo 2023, Cuadro 1 (no está en el paquete por corregimiento)
republica = {"viviendas": 1201809, "piso_tierra": 65379, "sin_agua_potable": 60679, "sin_luz": 58032,
             "cocinan_lena": 72179, "sin_internet": 363721}
areas = {"República": republica, "Provincia de Coclé": t["viviendas_totales"].set_index("nivel").loc["provincia"],
         "Distrito de Penonomé": distrito}
items = {"piso_tierra": "Piso de tierra", "sin_agua_potable": "Sin agua potable", "sin_luz": "Sin luz eléctrica",
         "cocinan_lena": "Cocinan con leña", "sin_internet": "Sin internet"}
carencias = pd.DataFrame({a: list(map(lambda k: tasa(v[k], v["viviendas"]), items)) for a, v in areas.items()},
                         index=list(items.values()))
print(carencias)

fig, ax = figura(10, 5.2)
y = range(len(carencias))
for i, (area, color) in enumerate(zip(carencias.columns, (GRIS, AZUL, NARANJA))):
    pos = [k + (i - 1) * 0.27 for k in y]
    ax.barh(pos, carencias[area], height=0.25, color=color, label=area)
    for p, v in zip(pos, carencias[area]):
        ax.text(v + 0.4, p, f"{v:.1f}", va="center", fontsize=8.5, color=TINTA2)
ax.set_yticks(list(y), carencias.index)
ax.invert_yaxis()
ax.set_xlim(0, 52)
ax.legend(frameon=False, loc="upper right")
estilo(ax, "El distrito arrastra más carencias que el país", "INEC, Censo 2023, Cuadro 1. Porcentajes calculados.",
       xlabel="% de viviendas particulares ocupadas", rejilla="x")
guardar(fig, "pulso-02-carencias")
plt.show()

verificar("en el distrito 41.9 % de las viviendas no tiene internet", carencias.loc["Sin internet", "Distrito de Penonomé"], 41.9)
verificar("en el país 30.3 % no tiene internet", carencias.loc["Sin internet", "República"], 30.3)

                   República  Provincia de Coclé  Distrito de Penonomé
Piso de tierra           5.4                 5.8                   9.3
Sin agua potable         5.0                 3.8                   5.9
Sin luz eléctrica        4.8                 7.0                   7.7
Cocinan con leña         6.0                10.8                  15.7
Sin internet            30.3                44.0                  41.9


Correcto: en el distrito 41.9 % de las viviendas no tiene internet
Correcto: en el país 30.3 % no tiene internet


/var/folders/xp/xx59bb5s1n31zc4kh2lnljyc0000gn/T/ipykernel_70040/2158287081.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** en las cinco carencias del gráfico el distrito está peor que el país; la brecha más grande es internet (41.9 % frente a 30.3 %).

### 5.3 ¿Cómo venía el empleo antes del sismo?

La encuesta de empleo no publica tasas por distrito, así que se usa la de Coclé. El desempleo pasó de 4.0 % en agosto de 2023 a 7.1 % en septiembre de 2025, después del cierre de la mina de cobre (noviembre de 2023), y bajó a 5.5 % en agosto de 2026, el último dato antes del sismo.

**Lo aprendido:** series temporales con índice de fechas ([unidad 38](../../01-python/semana-06/clase-1-ordenacion-y-series-temporales/u38-series-temporales.ipynb)).

In [6]:
serie = t["desempleo"].assign(fecha=lambda x: pd.to_datetime(x["fecha"])).set_index("fecha")
print(serie[["tasa_cocle", "tasa_republica", "desocupados_cocle"]])

fig, ax = figura(10, 5.2)
ax.plot(serie.index, serie["tasa_republica"], color=GRIS, linewidth=2, marker="o", label="República")
ax.plot(serie.index, serie["tasa_cocle"], color=AZUL, linewidth=2.5, marker="o", markersize=7, label="Provincia de Coclé")
for f, v in serie["tasa_cocle"].items():
    ax.annotate(f"{v:.1f} %", (f, v), textcoords="offset points", xytext=(0, -16), ha="center", fontsize=9, color=AZUL)
ax.axvline(pd.Timestamp("2023-11-28"), color=NARANJA, linewidth=1.2, linestyle="--")
ax.text(pd.Timestamp("2023-12-15"), 11.6, "Cierre de la mina\n(nov-2023)", fontsize=9, color=NARANJA, va="top")
ax.axvline(pd.Timestamp("2026-10-09"), color=NARANJA, linewidth=1.2, linestyle="--")
ax.text(pd.Timestamp("2026-09-01"), 11.6, "Sismo\n(9-oct-2026)", fontsize=9, color=NARANJA, va="top", ha="right")
ax.set_ylim(0, 12)
ax.legend(frameon=False, loc="upper left")
estilo(ax, "Desempleo de Coclé: 4.0 % en 2023, 7.1 % en 2025 y 5.5 % antes del sismo",
       "INEC, Encuesta de Mercado Laboral, Cuadro 2 (2019-2025) e informe de agosto de 2026.", ylabel="% de la PEA")
guardar(fig, "pulso-03-desempleo")
plt.show()

cambio = serie.loc["2025-09", "desocupados_cocle"].iloc[0] - serie.loc["2023-08", "desocupados_cocle"].iloc[0]
verificar("de 2023 a 2025 Coclé sumó 4,391 desocupados", int(cambio), 4391)

            tasa_cocle  tasa_republica  desocupados_cocle
fecha                                                    
2019-08-01         4.8             7.1               6528
2021-10-01         5.9            11.3               7992
2022-04-01         5.5             9.9               7736
2023-08-01         4.0             7.4               5774
2024-10-01         5.7             9.7               8031
2025-09-01         7.1            10.4              10165
2026-08-01         5.5             8.5               7771
Correcto: de 2023 a 2025 Coclé sumó 4,391 desocupados


/var/folders/xp/xx59bb5s1n31zc4kh2lnljyc0000gn/T/ipykernel_70040/1942379377.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** el desempleo de Coclé subió de 4.0 % a 7.1 % tras el cierre de la mina y bajó a 5.5 % en agosto de 2026, con más informalidad.

### 5.4 ¿Cómo es el empleo de Coclé hoy?

La encuesta de agosto de 2026 muestra quién está sin trabajo y cómo es el trabajo que hay. Usa un marco muestral nuevo, así que se comparan tasas, no niveles.

**Lo aprendido:** filtrar un DataFrame con condiciones y `set_index` ([unidad 36](../../01-python/semana-05/clase-2-modulos-y-pandas/u36-dataframes-de-pandas.ipynb)).

In [7]:
e = t["eml_2026"]
tasa_de = lambda ind, area: float(e[(e["indicador"] == ind) & (e["area"] == area)]["valor"].iloc[0])
barras = {"Desempleo total": tasa_de("Tasa de desempleo", "Coclé"),
          "Desempleo de mujeres": tasa_de("Tasa de desempleo", "Coclé, mujeres"),
          "Desempleo de hombres": tasa_de("Tasa de desempleo", "Coclé, hombres"),
          "Desempleo urbano": tasa_de("Tasa de desempleo", "Coclé urbana"),
          "Desempleo rural": tasa_de("Tasa de desempleo", "Coclé rural"),
          "Subempleo visible": tasa_de("Subempleo visible", "Coclé"),
          "Subempleo invisible": tasa_de("Subempleo invisible", "Coclé"),
          "Informalidad no agrícola": tasa_de("Informalidad no agrícola", "Coclé")}
serie26 = pd.Series(barras)
print(serie26)

fig, ax = figura(10, 5.0)
colores = [AZUL] * 5 + [NARANJA] * 3
ax.barh(serie26.index[::-1], serie26.values[::-1], color=colores[::-1], height=0.65)
for i, v in enumerate(serie26.values[::-1]):
    ax.text(v + 0.6, i, f"{v:.1f} %", va="center", fontsize=9.5, color=TINTA2)
ax.set_xlim(0, 70)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=AZUL), plt.Rectangle((0, 0), 1, 1, color=NARANJA)],
          labels=["Desempleo (% de la PEA)", "Calidad del empleo (% de ocupados)"], frameon=False, loc="upper right")
estilo(ax, "Coclé, agosto de 2026: poco desempleo, mucho empleo precario",
       "INEC, Encuesta de Mercado Laboral agosto 2026 (P), cuadros 3, 6 y 26, e informe de la población ocupada, cuadro 3.",
       xlabel="%", rejilla="x")
guardar(fig, "pulso-10-empleo-2026")
plt.show()
ocu = tasa_de("Población ocupada", "Coclé"); desoc = tasa_de("Población desocupada", "Coclé"); pea = tasa_de("Población económicamente activa", "Coclé")
verificar("ocupados + desocupados = PEA en agosto de 2026", ocu + desoc, pea)
verificar("la tasa de desempleo coincide con desocupados / PEA", tasa(desoc, pea), 5.5)

Desempleo total              5.5
Desempleo de mujeres         8.2
Desempleo de hombres         3.7
Desempleo urbano             6.0
Desempleo rural              5.1
Subempleo visible           12.4
Subempleo invisible         20.0
Informalidad no agrícola    59.5
dtype: float64


Correcto: ocupados + desocupados = PEA en agosto de 2026
Correcto: la tasa de desempleo coincide con desocupados / PEA


/var/folders/xp/xx59bb5s1n31zc4kh2lnljyc0000gn/T/ipykernel_70040/3215177469.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** el desempleo de las mujeres (8.2 %) duplica al de los hombres (3.7 %), y seis de cada diez empleos fuera del agro son informales. Uno de cada cinco ocupados tiene subempleo invisible: trabaja, pero gana poco para las horas que trabaja.

### 5.5 ¿De qué vive Coclé?

El PIB de Coclé de 2024 por actividad muestra qué sectores sostienen la economía provincial y, por lo tanto, cuáles quedan expuestos cuando un sismo golpea comercios, hoteles y caminos.

**Lo aprendido:** ordenar un DataFrame con `sort_values` ([unidad 37](../../01-python/semana-06/clase-1-ordenacion-y-series-temporales/u37-ordenacion-de-datos.ipynb)).

In [8]:
pib = t["pib"].set_index("actividad")["2024"]
total = pib.loc["PIB COCLÉ"]
actividades = pib.drop(["VALOR AGREGADO BRUTO", "Impuestos netos a los productos", "PIB COCLÉ", "PIB REPÚBLICA"])
top = actividades.sort_values(ascending=False).head(7)
print(top)

fig, ax = figura(10, 5.0)
ax.barh(top.index[::-1], top.values[::-1], color=AZUL, height=0.65)
for i, v in enumerate(top.values[::-1]):
    ax.text(v + 4, i, f"{v:,.1f}  ({v / total * 100:.0f} %)", va="center", fontsize=9, color=TINTA2)
ax.set_xlim(0, top.max() * 1.3)
estilo(ax, "¿De qué vive Coclé? PIB 2024 por actividad",
       "INEC, PIB provincial 2018-2024, millones de balboas en volumen encadenado; 2024 estimado.",
       xlabel="Millones de B/.", rejilla="x")
guardar(fig, "pulso-04-economia")
plt.show()
print(f"PIB de Coclé 2024: B/. {total:,.1f} millones")

actividad
Construcción                                     475.4
Otra producción no de mercado (gobierno)         386.1
Industrias manufactureras                        306.7
Agricultura ganadería y pesca                    278.2
Inmobiliarias profesionales y administrativas    195.8
Comercio (incl. zonas francas)                   138.2
Electricidad agua y saneamiento                  129.2
Name: 2024, dtype: float64


PIB de Coclé 2024: B/. 2,413.7 millones


/var/folders/xp/xx59bb5s1n31zc4kh2lnljyc0000gn/T/ipykernel_70040/1971755121.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. ¿Qué pasó el 9 de octubre de 2026?

### 6.1 ¿Cuántas viviendas resultaron afectadas?

Los comunicados de SINAPROC suman lo evaluado hasta cada corte: en cuatro días el número de viviendas afectadas en Coclé se multiplicó por nueve. El comunicado #12 (14 de octubre) es el último con desglose provincial y era preliminar.

**Lo aprendido:** variación entre cortes con `pct_change` ([unidad 38](../../01-python/semana-06/clase-1-ordenacion-y-series-temporales/u38-series-temporales.ipynb)).

In [9]:
danos = t["danos"].assign(fecha=lambda x: pd.to_datetime(x["fecha_hora"]))
danos["crecimiento_pct"] = (danos["viviendas_afectadas"].pct_change() * 100).round(0)
print(danos[["corte", "fecha", "viviendas_afectadas", "crecimiento_pct"]])

fig, ax = figura(10, 4.8)
etiquetas = [f"{c}\n{f:%d-%b}".replace("Oct", "oct") for c, f in zip(danos["corte"], danos["fecha"])]
barras = ax.bar(etiquetas, danos["viviendas_afectadas"], color=[AZUL, AZUL, NARANJA], width=0.55)
for b in barras:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 25, f"{int(b.get_height()):,}", ha="center", fontsize=12,
            fontweight="bold", color=TINTA2)
ax.set_ylim(0, 1550)
estilo(ax, "Viviendas afectadas en Coclé: de 149 a 1,324 en cuatro días",
       "SINAPROC, comunicados #5, #7 y #12. El #12 es un balance preliminar.", ylabel="Viviendas afectadas")
guardar(fig, "pulso-05-danos")
plt.show()
verificar("el último balance es 8.9 veces el primero", round(danos["viviendas_afectadas"].iloc[-1] / danos["viviendas_afectadas"].iloc[0], 1), 8.9)

                         corte               fecha  viviendas_afectadas  \
0                Comunicado #5 2026-10-10 18:30:00                  149   
1                Comunicado #7 2026-10-11 12:00:00                  229   
2  Comunicado #12 (preliminar) 2026-10-14 12:00:00                 1324   

   crecimiento_pct  
0              NaN  
1             54.0  
2            478.0  


Correcto: el último balance es 8.9 veces el primero


/var/folders/xp/xx59bb5s1n31zc4kh2lnljyc0000gn/T/ipykernel_70040/3371525949.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** 1,324 viviendas afectadas y 4,703 personas en Coclé al 14 de octubre, en un balance todavía preliminar.

### 6.2 ¿Cuánto duraron las réplicas?

El Instituto de Geociencias de la Universidad de Panamá localizó unas 1,690 réplicas hasta el 20 de noviembre, ya menos de 10 por día. La gente siguió durmiendo afuera durante semanas.

**Lo aprendido:** frecuencia diaria a partir de una serie acumulada con `diff` ([unidad 38](../../01-python/semana-06/clase-1-ordenacion-y-series-temporales/u38-series-temporales.ipynb)).

In [10]:
rep = t["replicas"].assign(fecha=lambda x: pd.to_datetime(x["fecha"])).set_index("fecha")["replicas_acumuladas"]
por_dia = (rep.diff() / rep.index.to_series().diff().dt.days).round(1)
print(pd.DataFrame({"acumuladas": rep, "réplicas por día en el tramo": por_dia}))

fig, ax = figura(10, 4.8)
ax.plot(rep.index, rep.values, color=AZUL, linewidth=2.5, marker="o", markersize=7)
for f, v in rep.items():
    ax.annotate(f"{v:,}", (f, v), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=9, color=TINTA2)
ax.set_ylim(0, 1900)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{pd.Timestamp(plt.matplotlib.dates.num2date(x)).day}-"
                                               f"{['oct', 'nov'][pd.Timestamp(plt.matplotlib.dates.num2date(x)).month - 10]}"))
dias = (rep.index[-1] - pd.Timestamp("2026-10-09")).days
estilo(ax, f"Unas {rep.iloc[-1]:,} réplicas en {dias} días; el ritmo bajó de {por_dia.iloc[1]:.0f} a {por_dia.iloc[-1]:.0f} por día",
       "IGC-UP, según la prensa (Panamá América, Infobae, Destino Panamá, Prensa Latina, Crítica, La Verdad, La Prensa). Réplicas por día: cálculo propio.",
       ylabel="Réplicas acumuladas")
guardar(fig, "pulso-06-replicas")
plt.show()

            acumuladas  réplicas por día en el tramo
fecha                                               
2026-10-10         135                           NaN
2026-10-16         746                         101.8
2026-11-02        1271                          30.9
2026-11-06        1452                          45.2
2026-11-09        1483                          10.3
2026-11-17        1656                          21.6
2026-11-20        1690                          11.3


/var/folders/xp/xx59bb5s1n31zc4kh2lnljyc0000gn/T/ipykernel_70040/96570017.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6.3 ¿Cómo está Penonomé hoy?

Un mes y medio después del sismo, estos son los datos más recientes que existen. SINAPROC no publicó un balance de Coclé posterior al 14 de octubre, así que varias cifras son nacionales o de la región afectada.

**Lo aprendido:** ordenar y filtrar un DataFrame ([unidad 37](../../01-python/semana-06/clase-1-ordenacion-y-series-temporales/u37-ordenacion-de-datos.ipynb)).

In [11]:
hoy = t["situacion"].sort_values("fecha", ascending=False)
cocle = hoy[hoy["area"].str.contains("Coclé|Penonomé")]
pd.set_option("display.max_colwidth", 60)
print(cocle[["fecha", "indicador", "valor", "unidad", "tipo_fuente"]].to_string(index=False))
proy = t["poblacion_proy"]
pen = proy[proy["area"] == "Penonomé (distrito)"].set_index("fecha")["poblacion"]
print()
print("Población del distrito de Penonomé:", pen.to_dict())
verificar("la proyección de 2024 coincide con la que ya usaba el estudio", int(pen.loc[2024]), 111457)

     fecha                                                 indicador     valor          unidad           tipo_fuente
2026-11-23                  Días sin cortes de luz por falta de pago     60.00            días               oficial
2026-11-17                       Estudiantes sin clases presenciales 193000.00     estudiantes prensa (cita oficial)
2026-11-09                      Viviendas en construcción del Miviot     67.00       viviendas               oficial
2026-11-09                 Viviendas contratadas para reconstrucción    439.00       viviendas prensa (cita oficial)
2026-11-09                            Monto de contratos de vivienda     12.68 millones de B/. prensa (cita oficial)
2026-11-06                     Fichas sociales de emergencia (FIBES)   1890.00          fichas               oficial
   2026-09                              Superficie de arroz sembrada   4097.84       hectáreas               oficial
2026-08-18 Contrataciones de Cobre Panamá en actividades autoriz

**Respuesta:** la reconstrucción avanza lento. Hay 439 viviendas contratadas en Coclé y 67 en obra; 1,890 familias tienen ficha social del MIDES; el mercado volvió a su edificio y el hospital recuperó la luz el 20 de noviembre. Las escuelas dañadas recién empiezan a demolerse. El distrito sigue creciendo: 113,409 habitantes proyectados para 2026.

## 7. ¿Qué viene para el empleo?

### 7.1 ¿Qué enseña el sismo de Colombia?

El 10 de agosto de 2026 un sismo de magnitud 7.4 golpeó el Eje Cafetero colombiano. Con la encuesta de empleo del DANE se compara cómo cambió el desempleo en las ciudades afectadas (Quibdó, Cali, Pereira, Armenia y Manizales) frente a ciudades de control lejos del epicentro. Si las afectadas empeoran más que las de control, la diferencia es una estimación del efecto del sismo (diferencias en diferencias).

**Lo aprendido:** `groupby` y `unstack` ([unidad 37](../../01-python/semana-06/clase-1-ordenacion-y-series-temporales/u37-ordenacion-de-datos.ipynb)).

In [12]:
orden = ["abr-jun", "may-jul", "jun-ago", "jul-sep"]
g = t["geih"][t["geih"]["grupo"].isin(["afectada", "control"])]
dd = g.groupby(["trimestre_movil", "grupo"])["cambio_pp"].mean().unstack().loc[orden]
dd["brecha"] = dd["afectada"] - dd["control"]
print(dd.round(2))

fig, ax = figura(10, 4.8)
x = range(len(orden))
ax.plot(x, dd["control"], color=GRIS, linewidth=2, marker="o", label="Ciudades de control")
ax.plot(x, dd["afectada"], color=NARANJA, linewidth=2.5, marker="o", markersize=7, label="Ciudades afectadas")
ax.axhline(0, color="#c3c2b7", linewidth=1)
ax.axvspan(1.5, 3.4, color="#f6e3da", alpha=0.6, lw=0)
ax.text(2.45, ax.get_ylim()[1] * 0.85 if ax.get_ylim()[1] > 0 else 0.5, "Trimestres que incluyen el sismo", ha="center", fontsize=9, color=NARANJA)
ax.set_xticks(list(x), orden)
ax.legend(frameon=False, loc="lower right")
estilo(ax, "Colombia: tras el sismo, las ciudades afectadas alcanzaron a las de control",
       "DANE, GEIH 2025 y 2026; cambio frente al mismo trimestre de 2025.", ylabel="Cambio del desempleo (puntos)")
guardar(fig, "pulso-07-colombia")
plt.show()
verificar("la brecha pasa de -1.87 a +0.18 puntos", (round(dd["brecha"].iloc[0], 2), round(dd["brecha"].iloc[-1], 2)), (-1.87, 0.18))

grupo            afectada  control  brecha
trimestre_movil                           
abr-jun             -1.58     0.29   -1.87
may-jul             -0.80     0.00   -0.80
jun-ago              0.08     0.50   -0.42
jul-sep              0.80     0.62    0.18


Correcto: la brecha pasa de -1.87 a +0.18 puntos


/var/folders/xp/xx59bb5s1n31zc4kh2lnljyc0000gn/T/ipykernel_70040/3300256877.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** en Colombia el golpe al empleo apareció con rezago, en los trimestres posteriores al sismo; en Coclé hay que esperar algo parecido.

### 7.2 ¿Qué escenarios hay para Coclé?

Con la línea base de 5.5 % (agosto de 2026) y la lección de Colombia se proyectan tres escenarios hasta 2030. **Son supuestos, no pronósticos:** sirven para dimensionar el riesgo mientras llega la encuesta de 2027.

**Lo aprendido:** programación dinámica con memoización para la recuperación año a año ([unidad 32](../../01-python/semana-05/clase-1-programacion-dinamica-y-backtracking/u32-programacion-dinamica.ipynb)); el detalle está en el anexo `notebooks/anexo-semanas/s05`.

In [13]:
esc = t["escenarios"].set_index("anio")
print(esc)
fig, ax = figura(10, 4.8)
for (nombre, color) in zip(esc.columns, (AGUA, AZUL, NARANJA)):
    ax.plot(esc.index, esc[nombre], color=color, linewidth=2.5, marker="o", label=nombre)
ax.scatter([2025.67], [5.5], color=TINTA2, zorder=3)
ax.annotate("Agosto 2026: 5.5 %", (2025.67, 5.5), textcoords="offset points", xytext=(8, -14), fontsize=9, color=TINTA2)
ax.set_ylim(4, 9.5)
ax.set_xticks(esc.index)
ax.legend(frameon=False, loc="upper right")
estilo(ax, "Tres escenarios de desempleo para Coclé (modelo, no dato oficial)",
       "Modelo del proyecto con línea base INEC (ago-2026) y la reacción observada en Colombia.", ylabel="% de la PEA")
guardar(fig, "pulso-08-escenarios")
plt.show()

      Rápido (como Colombia nacional)  Base  Lento (norte rural, informal)
anio                                                                      
2026                             6.33  7.40                           8.49
2027                             5.69  6.60                           7.92
2028                             5.51  5.94                           7.21
2029                             5.50  5.67                           6.71
2030                             5.50  5.57                           6.36


/var/folders/xp/xx59bb5s1n31zc4kh2lnljyc0000gn/T/ipykernel_70040/3718475454.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.3 ¿Qué pasa si no se resuelve el problema?

El escenario lento es lo que pasaría sin una respuesta que llegue al norte rural e informal; el rápido, con una recuperación como la de Colombia. Para pasar de tasas a personas se usa la fuerza laboral de Coclé de agosto de 2026 (142,130 personas), que se mantiene fija: es un supuesto del modelo, no un dato.

**Lo aprendido:** operaciones entre columnas de un DataFrame y `cumsum` ([unidad 36](../../01-python/semana-05/clase-2-modulos-y-pandas/u36-dataframes-de-pandas.ipynb)).

In [14]:
PEA = 142130          # INEC, agosto 2026: se mantiene fija (supuesto)
BASE_DESOCUPADOS = 7771
sin = esc["Lento (norte rural, informal)"]
con = esc["Rápido (como Colombia nacional)"]
personas = pd.DataFrame({"sin_intervencion": (sin / 100 * PEA).round(), "con_intervencion": (con / 100 * PEA).round()}).astype(int)
personas["diferencia"] = personas["sin_intervencion"] - personas["con_intervencion"]
personas["diferencia_acumulada"] = personas["diferencia"].cumsum()
print(personas)

fig, ax = figura(10, 5.0)
ax.plot(personas.index, personas["sin_intervencion"], color=NARANJA, linewidth=2.5, marker="o", label="Sin intervención (escenario lento)")
ax.plot(personas.index, personas["con_intervencion"], color=AGUA, linewidth=2.5, marker="o", label="Con intervención (escenario rápido)")
ax.fill_between(personas.index, personas["con_intervencion"], personas["sin_intervencion"], color=NARANJA, alpha=0.12)
ax.axhline(BASE_DESOCUPADOS, color=GRIS, linewidth=1.2, linestyle="--")
ax.text(personas.index[-1], BASE_DESOCUPADOS - 450, "Antes del sismo: 7,771", ha="right", fontsize=9, color=TINTA2)
for x, y in personas["sin_intervencion"].items():
    ax.annotate(f"{y:,}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9, color=NARANJA)
ax.set_ylim(6000, 13500)
ax.set_xticks(personas.index)
ax.legend(frameon=False, loc="upper right")
estilo(ax, "Si no se resuelve: hasta 4,300 desocupados más en Coclé en 2026",
       "Modelo del proyecto (no es dato oficial): escenarios de desempleo × PEA de Coclé de agosto 2026 (INEC), fija.",
       ylabel="Personas desocupadas")
guardar(fig, "pulso-11-sin-intervencion")
plt.show()

extra_2026 = int(personas.loc[2026, "sin_intervencion"] - BASE_DESOCUPADOS)
acumulado = int(personas["diferencia_acumulada"].iloc[-1])
vuelve = [a for a, v in sin.items() if v <= 5.5]
print(f"Desocupados de más en 2026 frente a antes del sismo: {extra_2026:,}")
print(f"Años-persona de desempleo de más de 2026 a 2030 sin intervención: {acumulado:,}")
print("Año en que el escenario lento vuelve a 5.5 %:", vuelve[0] if vuelve else "no vuelve antes de 2031")
verificar("cada escenario convierte la tasa en personas con la misma PEA", int(round(sin.loc[2026] / 100 * PEA)), int(personas.loc[2026, "sin_intervencion"]))
verificar("el escenario lento no vuelve a 5.5 % antes de 2031", vuelve, [])

      sin_intervencion  con_intervencion  diferencia  diferencia_acumulada
anio                                                                      
2026             12067              8997        3070                  3070
2027             11257              8087        3170                  6240
2028             10248              7831        2417                  8657
2029              9537              7817        1720                 10377
2030              9039              7817        1222                 11599


Desocupados de más en 2026 frente a antes del sismo: 4,296
Años-persona de desempleo de más de 2026 a 2030 sin intervención: 11,599
Año en que el escenario lento vuelve a 5.5 %: no vuelve antes de 2031
Correcto: cada escenario convierte la tasa en personas con la misma PEA
Correcto: el escenario lento no vuelve a 5.5 % antes de 2031


/var/folders/xp/xx59bb5s1n31zc4kh2lnljyc0000gn/T/ipykernel_70040/3659011125.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** sin una respuesta que llegue al norte, Coclé tendría unos 12,000 desocupados en 2026 (4,300 más que antes del sismo) y todavía 9,000 en 2030, sin volver al 5.5 %. Frente a una recuperación rápida, son unos 11,600 años-persona de desempleo de más en cinco años. Es un modelo con supuestos, no una medición.

## 8. ¿Dónde ayudar primero?

Este es el resultado que se vende: un **índice de prioridad** de 0 a 100 para cada corregimiento, que combina:

- **Pobreza** (50 %): pobreza multidimensional por corregimiento 2023.
- **Carencias de la vivienda** (30 %): promedio de viviendas con piso de tierra, sin agua potable y sin luz, escalado al corregimiento con más carencias.
- **Daños reportados** (20 %): si hubo al menos un registro de daños. Se usa sí o no, y no la cantidad, porque la prensa cubre mucho más la cabecera y contar registros premiaría la visibilidad y no la necesidad.

**Lo aprendido:** funciones con parámetros por defecto (los pesos), ordenamiento por fusión para el ranking ([unidad 28](../../01-python/semana-04/clase-1-algoritmos-de-ordenamiento/u28-ordenamiento-por-fusion.ipynb)) y búsqueda binaria para ubicar un umbral ([unidad 25](../../01-python/semana-03/clase-2-busqueda-binaria-tablas-hash/u25-busqueda-binaria.ipynb)).

In [15]:
corr["prioridad"] = pulso.indice_prioridad(corr)
ranking, _ = fusion(corr.to_dict("records"), clave=lambda c: -c["prioridad"])
tabla = pd.DataFrame(ranking)[["corregimiento", "prioridad", "ipm", "carencias_pct", "registros_danos", "pob_2023"]]
tabla.index = range(1, len(tabla) + 1)
print(tabla.to_string())

# ¿Cuántos corregimientos superan 50 puntos? Búsqueda binaria sobre los puntajes ordenados de menor a mayor
puntajes = sorted(corr["prioridad"])
pos = next(i for i, p in enumerate(puntajes) if p >= 50)
verificar("la búsqueda binaria encuentra el primer puntaje del umbral", bin_search(puntajes, puntajes[pos]), pos)
altos = tabla[tabla["prioridad"] >= 50]
print(f"Corregimientos con prioridad alta (50 o más): {len(altos)}, con {altos['pob_2023'].sum():,} habitantes")

                 corregimiento  prioridad   ipm  carencias_pct  registros_danos  pob_2023
1                         Tulú       85.4  76.0           33.3                1      2241
2                   San Miguel       83.2  81.0           27.5                1      2038
3                      Riecito       73.0  86.0           36.4                0      1315
4              Chiguirí Arriba       66.0  64.0           17.0                1      7185
5                    Río Indio       65.9  77.0           33.3                0      4026
6            Candelario Ovalle       60.4  75.0           27.8                0      3615
7                Boca de Tucué       58.8  71.0           28.3                0      2090
8                       Toabré       54.9  47.0           13.8                6      7440
9                      Pajonal       51.6  45.0           11.1                1      7678
10                   Las Minas       50.8  47.0            8.8                1      2662
11        

In [16]:
# Sensibilidad: ¿cambia el grupo de prioridad alta si todos los criterios pesan lo mismo?
iguales = pulso.indice_prioridad(corr, {"pobreza": 1, "carencias": 1, "danos": 1})
top_base = set(tabla.head(6)["corregimiento"])
top_igual = set(corr.assign(p=iguales).nlargest(6, "p")["corregimiento"])
print("Seis primeros con los pesos del proyecto:", sorted(top_base))
print("Seis primeros con pesos iguales:        ", sorted(top_igual))
nucleo = sorted(top_base & top_igual)
print("Coinciden:", len(nucleo), "de 6 ->", nucleo)
norte = set(corr.loc[corr["ipm"] >= 45, "corregimiento"])
verificar("con ambos pesos los seis primeros son del norte pobre (pobreza de 45 % o más)", (top_base | top_igual) <= norte, True)
verificar("Tulú, San Miguel, Riecito y Chiguirí Arriba encabezan con ambos pesos", nucleo,
          ["Chiguirí Arriba", "Riecito", "San Miguel", "Tulú"])

Seis primeros con los pesos del proyecto: ['Candelario Ovalle', 'Chiguirí Arriba', 'Riecito', 'Río Indio', 'San Miguel', 'Tulú']
Seis primeros con pesos iguales:         ['Chiguirí Arriba', 'Pajonal', 'Riecito', 'San Miguel', 'Toabré', 'Tulú']
Coinciden: 4 de 6 -> ['Chiguirí Arriba', 'Riecito', 'San Miguel', 'Tulú']
Correcto: con ambos pesos los seis primeros son del norte pobre (pobreza de 45 % o más)
Correcto: Tulú, San Miguel, Riecito y Chiguirí Arriba encabezan con ambos pesos


In [17]:
fig, ax = figura(10, 6.0)
orden = tabla.iloc[::-1]
colores = [NARANJA if p >= 50 else AZUL for p in orden["prioridad"]]
barras = ax.barh(orden["corregimiento"], orden["prioridad"], color=colores, height=0.7)
for b, (_, f) in zip(barras, orden.iterrows()):
    dano = "con daños reportados" if f["registros_danos"] > 0 else "sin registros de daños"
    ax.text(b.get_width() + 1, b.get_y() + b.get_height() / 2, f"{f['prioridad']:.0f}   pobreza {f['ipm']:.0f} %, {dano}",
            va="center", fontsize=8.5, color=TINTA2)
ax.set_xlim(0, 135)
ax.axvline(50, color="#c3c2b7", linewidth=1, linestyle="--")
estilo(ax, f"Dónde ayudar primero: {len(altos)} corregimientos del norte con prioridad alta",
       "Índice del proyecto: pobreza (IPM 2023) 50 %, carencias de vivienda (Censo 2023) 30 %, daños reportados 20 %.",
       xlabel="Índice de prioridad (0 a 100)", rejilla="x")
guardar(fig, "pulso-09-prioridad")
plt.show()

/var/folders/xp/xx59bb5s1n31zc4kh2lnljyc0000gn/T/ipykernel_70040/157470836.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Respuesta:** diez corregimientos del norte, con 40,290 habitantes, tienen prioridad alta; Tulú, San Miguel, Riecito y Chiguirí Arriba encabezan con cualquier peso.

## 9. Resumen de hallazgos

Ordenados por su importancia para decidir:

1. **La ayuda tiene que ir primero al norte.** Diez corregimientos con 40,290 habitantes tienen prioridad alta; Tulú, San Miguel, Riecito y Chiguirí Arriba encabezan con cualquier combinación de pesos.
2. **La prensa mira la cabecera y la necesidad está en otra parte.** La cabecera tiene 28 de los 43 registros de daños, pero pobreza de 16 %; en el norte hay pocos registros y pobreza de hasta 86 %.
3. **El sismo golpeó a un distrito que ya estaba frágil:** más carencias que el país y un empleo que venía del cierre de la mina (4.0 % a 7.1 %).
4. **El impacto en el empleo todavía no se ve,** pero llegará: en Colombia apareció con rezago. Para Coclé, 2026 podría cerrar entre 6.3 % y 8.5 %; si no se resuelve, serían unos 4,300 desocupados más y no se volvería a 5.5 % antes de 2031.
5. **Los datos copiados también fallan:** la limpieza encontró un valor de Colón en la fila de Coclé. Verificar antes de analizar cambia los resultados.

### Limitaciones y ética

- No existe todavía una encuesta de empleo posterior al sismo: los escenarios son supuestos, no mediciones.
- Los daños por corregimiento salen de una compilación de prensa: son registros, no unidades dañadas, y faltan las zonas aisladas.
- El índice de prioridad depende de los pesos elegidos; por eso se probó con otros pesos y se informa qué parte del ranking es estable.
- Los datos son agregados, sin información personal. Señalar corregimientos pobres busca priorizar la ayuda, no estigmatizarlos.

**Recomendaciones:** validar el índice en campo con el municipio y SINAPROC, pedir el desglose de daños por corregimiento y repetir este análisis con un comando cuando salga la encuesta de 2027.

[Proyecto](README.md) | [Estudio completo](documento/estudio-penonome.md) | [Anexo: técnicas por semana](notebooks/anexo-semanas/s01-listas-diccionarios-conjuntos.ipynb)